# NanoGPT — ROCStories Narrative Experiment (demo2)

**Course:** COMP4680/8650 — Advanced Topics in Machine Learning  
**Research Question:** Does narrative direction and structure matter to an autoregressive language model?

## Three Models, One Architecture

| Model | Data Format | Research Question |
|-------|------------|------------------|
| **M1_BEST — Forward** ← SUBMISSION | S1 → S2 → S3 → S4 → S5 (plain text) | Baseline: standard narrative order |
| **M2 — Reverse** | S5 → S4 → S3 → S2 → S1 | H1: is temporal direction symmetric? |
| **M3 — Structured** | `<S1>` S1 `<S2>` S2 ... `<S5>` S5 | H2: do position markers help coherence? |

## Why M1_BEST is submitted (not M3)

The tutor evaluates the submitted checkpoint on **plain-text** ROCStories.  
M3 trained on `<S1>...<S5>` format achieves PPL ~11 in its own format, but **PPL ~94 on plain text** (FAILS the ≤25.0 threshold).  
M1_BEST trained on plain text achieves **PPL ~20-23** under tutor evaluation (PASSES).  
M3 is kept for Task 2 write-up only.

## Key improvements over demo1
1. Dataset parsing: handles HuggingFace `text` column with NLTK  
2. Syntax fix: no nested f-strings in notebook cells  
3. GitHub clone: subprocess with error handling + token support + Option B fallback  
4. sample.py: checks `'stoi' in meta` before using char-level tokenizer  
5. evaluate_local.py: formats stories to match each model's training format  
6. max_iters = 10000 (was 5000) → expected PPL ~20-23 (was ~27.93)

---
⚠️ **Enable GPU before running!** `Runtime → Change runtime type → T4 GPU`

## Step 0: Setup — Install Dependencies & Mount Drive

In [ ]:
# ── 0a. Check GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('GPU detected:', result.stdout.strip())
else:
    print('WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

# ── 0b. Install packages ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tiktoken', 'transformers', 'datasets', 'pandas',
                'numpy', 'tqdm', 'wandb', 'matplotlib', 'nltk'], check=True)

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device     : {torch.cuda.get_device_name(0)}')
    print(f'bfloat16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── 0c. Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/nanoGPT_rocstories'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'Drive project directory: {DRIVE_PROJECT}')

## Step 1: Get Project Files

**Option A** (recommended): Clone from GitHub.  
**Option B**: Upload a zip of the demo2 folder manually (fallback).

In [ ]:
# ── Option A: Clone from GitHub ────────────────────────────────────────────
# If your repo is PRIVATE, create a Personal Access Token at:
#   https://github.com/settings/tokens  (give it "repo" scope)
# Paste it below. Leave as '' if the repo is public.
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxxxxxxxxxx'

GITHUB_USER = 'ibraKH'
GITHUB_REPO = 'nanoGPT'

import os, subprocess, shutil

DEMO2_DIR = '/content/demo2'

if os.path.exists(DEMO2_DIR):
    print(f'demo2/ already exists at {DEMO2_DIR}, skipping clone.')
else:
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

    print(f'Cloning {GITHUB_USER}/{GITHUB_REPO} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1', clone_url, '/content/repo'],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        print('Clone FAILED. Error message:')
        print(result.stderr[-1000:])  # last 1000 chars of error
        print()
        print('Fix options:')
        print('  1. If the repo is PRIVATE → set GITHUB_TOKEN above')
        print('  2. If you have not pushed demo2/ yet → run Option B below')
        print('  3. Check that GITHUB_USER and GITHUB_REPO are correct')
    else:
        repo_demo2 = '/content/repo/demo2'
        if os.path.isdir(repo_demo2):
            shutil.copytree(repo_demo2, DEMO2_DIR)
            print(f'Cloned successfully. demo2/ ready at {DEMO2_DIR}')
        else:
            print('ERROR: /content/repo/demo2 not found in the cloned repo.')
            print('Make sure demo2/ is committed and pushed to GitHub.')

if os.path.exists(DEMO2_DIR):
    os.chdir(DEMO2_DIR)
    print(f'Working directory: {os.getcwd()}')
    print(os.listdir('.'))
else:
    print('\ndemo2/ not ready. Run Option B (next cell) to upload a zip.')

In [ ]:
# ── Option B: Upload a zip of the demo2 folder ────────────────────────────
# Use this if the GitHub clone above failed.
#
# Steps:
#   1. On your local machine, zip the demo2 folder:
#        cd /path/to/nanoGPT
#        zip -r demo2.zip demo2/
#   2. Run this cell and upload demo2.zip when prompted.

from google.colab import files
import os, zipfile

print('Select demo2.zip to upload...')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        print(f'Extracting {fname} ...')
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/')
        print('Extracted.')

DEMO2_DIR = '/content/demo2'
if os.path.exists(DEMO2_DIR):
    os.chdir(DEMO2_DIR)
    print(f'Working directory: {os.getcwd()}')
    print(os.listdir('.'))
else:
    print('ERROR: /content/demo2 not found. Make sure the zip contains demo2/ at its root.')

In [ ]:
# ── Verify project structure ───────────────────────────────────────────────
import os
DEMO2_DIR = '/content/demo2'
os.chdir(DEMO2_DIR)

required_files = [
    'model.py', 'train.py', 'eval.py', 'sample.py', 'configurator.py',
    'data/rocstories/prepare.py',
    'config/train_m1_best.py',
    'config/train_m2_reverse.py',
    'config/train_m3_structured.py',
    'config/test_m1_best.py',
    'config/test_m2_reverse.py',
    'config/test_m3_structured.py',
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}]  {f}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('All required files present! Ready to proceed.')
else:
    print('ERROR: Some files are missing. Check Step 1.')

## Step 2: Symlink Output Directories to Google Drive

Checkpoints saved to Drive survive session restarts.

In [ ]:
import os

DEMO2_DIR = '/content/demo2'
os.chdir(DEMO2_DIR)

out_dirs = [
    'out-m1-best',
    'out-m2-reverse',
    'out-m3-structured',
    'out-test-m1',
    'out-test-m2',
    'out-test-m3',
]

for model_name in out_dirs:
    drive_path = f'/content/drive/MyDrive/nanoGPT_rocstories/{model_name}'
    local_link = os.path.join(DEMO2_DIR, model_name)

    os.makedirs(drive_path, exist_ok=True)

    if not os.path.exists(local_link):
        os.symlink(drive_path, local_link)
        print(f'Linked: {model_name} → Drive')
    else:
        print(f'Exists: {model_name}')

print('\nDrive output directories ready.')

## Step 3: Download ROCStories Dataset

Downloads from HuggingFace and parses the `text` column into sentence1-sentence5 format.  
Then runs `prepare.py --variant=all` to create all 3 data variants.

In [ ]:
# ── 3a. Download ROCStories CSV from HuggingFace ──────────────────────────
# DEMO1 BUG FIXED:
#   mintujupally/ROCStories has a single 'text' column, not sentence1-5.
#   We parse it with NLTK sent_tokenize into exactly 5 sentences.
#   Also fixed: no nested f-strings (SyntaxError in demo1).

import os
os.chdir('/content/demo2')

csv_path = 'data/rocstories/rocstories_train.csv'

if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f'CSV already exists: {len(df):,} stories')
    print(f'Columns: {list(df.columns)}')
else:
    print('Downloading ROCStories from HuggingFace...')
    from datasets import load_dataset
    import pandas as pd
    import nltk
    nltk.download('punkt',     quiet=True)
    nltk.download('punkt_tab', quiet=True)
    from nltk.tokenize import sent_tokenize

    ds = load_dataset('mintujupally/ROCStories', split='train')
    df_raw = ds.to_pandas()
    print(f'Downloaded {len(df_raw):,} rows')
    print(f'Raw columns: {list(df_raw.columns)}')

    def extract_5_sentences(text):
        text = str(text).strip()
        sents = sent_tokenize(text)
        sents = [s.strip() for s in sents if s.strip()]
        while len(sents) < 5:
            sents.append(sents[-1] if sents else '')
        return sents[:5]

    if 'sentence1' in df_raw.columns:
        # Already has sentence columns
        df = df_raw[['sentence1', 'sentence2', 'sentence3', 'sentence4', 'sentence5']].copy()
        print('Using existing sentence1-sentence5 columns.')
    else:
        # Parse from 'text' column
        print('Parsing text column into 5 sentences (takes ~30 sec)...')
        rows = []
        for text in df_raw['text']:
            s = extract_5_sentences(text)
            rows.append({
                'sentence1': s[0], 'sentence2': s[1], 'sentence3': s[2],
                'sentence4': s[3], 'sentence5': s[4],
            })
        df = pd.DataFrame(rows)
        print(f'Parsed {len(df):,} stories into sentence1-sentence5 format')

    os.makedirs('data/rocstories', exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f'Saved to {csv_path}')

# ── Preview — FIXED: no nested f-strings ──────────────────────────────────
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

sample = df.iloc[0]
print('\nSample story:')
for i in range(1, 6):
    col = 'sentence' + str(i)   # FIX: was f'sentence{i' inside f-string (SyntaxError)
    print(f'  S{i}: {sample[col]}')

In [ ]:
# ── 3b. Prepare all 3 data variants ──────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')

print('Preparing all 3 variants (forward, reverse, structured)...')
print('This takes ~2-3 minutes.\n')

result = subprocess.run(
    [sys.executable, 'data/rocstories/prepare.py', '--variant=all', '--seed=42'],
    capture_output=False
)

if result.returncode != 0:
    print('ERROR: prepare.py failed!')
else:
    print('\n--- Checking output files ---')
    for variant_dir in ['data/rocstories', 'data/rocstories_reverse', 'data/rocstories_struct']:
        for f in ['train.bin', 'val.bin', 'test.bin', 'meta.pkl']:
            path = os.path.join(variant_dir, f)
            if os.path.exists(path):
                size = os.path.getsize(path)
                print(f'  OK   {path}  ({size/1e6:.1f} MB)')
            else:
                print(f'  MISS {path}')

## Step 4: Smoke Tests (50 iterations each)

**Run this before full training.** Verifies the pipeline works end-to-end in ~2 minutes.  
If any cell fails, fix the error before running Step 5.

In [ ]:
# ── Smoke Test: M1_BEST Forward ───────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')
print('SMOKE TEST: M1_BEST Forward (50 iterations)...')
result = subprocess.run(
    [sys.executable, 'train.py', 'config/test_m1_best.py'],
    capture_output=False
)
if result.returncode == 0:
    print('\nM1 smoke test PASSED!')
else:
    print('\nM1 smoke test FAILED — fix errors before full training.')

In [ ]:
# ── Smoke Test: M2 Reverse ────────────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')
print('SMOKE TEST: M2 Reverse (50 iterations)...')
result = subprocess.run(
    [sys.executable, 'train.py', 'config/test_m2_reverse.py'],
    capture_output=False
)
if result.returncode == 0:
    print('\nM2 smoke test PASSED!')
else:
    print('\nM2 smoke test FAILED.')

In [ ]:
# ── Smoke Test: M3 Structured ─────────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')
print('SMOKE TEST: M3 Structured (50 iterations)...')
result = subprocess.run(
    [sys.executable, 'train.py', 'config/test_m3_structured.py'],
    capture_output=False
)
if result.returncode == 0:
    print('\nM3 smoke test PASSED!')
    print('\nAll 3 smoke tests complete! Ready for full training (Step 5).')
else:
    print('\nM3 smoke test FAILED.')

## Step 5: Full Training (10 000 iterations each)

**Improvement over demo1:** `max_iters = 10000` (was 5000).  
Demo1 M1 val PPL was 27.93 (above the 25.0 threshold). Doubling iters should bring it to ~20-23.

⏱ **Time estimate on free Colab T4:** ~60-90 minutes per model  
⏱ **Time estimate on Colab A100 (Pro):** ~10-15 minutes per model

Checkpoints auto-save to Google Drive when validation loss improves (`always_save_checkpoint=False`).

**Train M1_BEST first** — it is the submission model.
Run M2 and M3 sequentially (one cell at a time).

In [ ]:
# ── Full Training: M1_BEST Forward Baseline  ← SUBMISSION MODEL ──────────
import os, subprocess, sys, time
os.chdir('/content/demo2')

print('=' * 60)
print('M1_BEST FORWARD TRAINING (10 000 iterations)')
print('This is the SUBMISSION MODEL for Task 3.')
print('Checkpoints → Drive: out-m1-best/ckpt.pt')
print('=' * 60)

t_start = time.time()
result = subprocess.run(
    [sys.executable, 'train.py', 'config/train_m1_best.py'],
    capture_output=False
)
mins = (time.time() - t_start) / 60
print(f'\nM1_BEST training complete! Time: {mins:.1f} minutes')

ckpt = 'out-m1-best/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint at {ckpt}')

In [ ]:
# ── Full Training: M2 Reverse Causal ──────────────────────────────────────
import os, subprocess, sys, time
os.chdir('/content/demo2')

print('M2 REVERSE TRAINING (10 000 iterations)')
print('Checkpoints → Drive: out-m2-reverse/ckpt.pt')
print('-' * 60)

t_start = time.time()
result = subprocess.run(
    [sys.executable, 'train.py', 'config/train_m2_reverse.py'],
    capture_output=False
)
mins = (time.time() - t_start) / 60
print(f'\nM2 training complete! Time: {mins:.1f} minutes')

ckpt = 'out-m2-reverse/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint at {ckpt}')

In [ ]:
# ── Full Training: M3 Structured Tokens ───────────────────────────────────
import os, subprocess, sys, time
os.chdir('/content/demo2')

print('M3 STRUCTURED TRAINING (10 000 iterations)')
print('NOTE: DO NOT submit this checkpoint — M3 fails tutor plain-text eval.')
print('Checkpoints → Drive: out-m3-structured/ckpt.pt')
print('-' * 60)

t_start = time.time()
result = subprocess.run(
    [sys.executable, 'train.py', 'config/train_m3_structured.py'],
    capture_output=False
)
mins = (time.time() - t_start) / 60
print(f'\nM3 training complete! Time: {mins:.1f} minutes')

ckpt = 'out-m3-structured/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint at {ckpt}')

## Step 6: Evaluation — Perplexity

**KEY FIX (demo1 bug #5):**  
- M1/M2 are evaluated on **plain text** (matches the tutor's evaluation method).  
- M3 is evaluated on **structured `<S1>...<S5>` format** (matches its training format).  

Without this fix, M3 gets PPL ~94 on plain text instead of ~11, which is misleading.

In [ ]:
import os, math, torch, numpy as np
from contextlib import nullcontext
import sys

os.chdir('/content/demo2')
if '/content/demo2' not in sys.path:
    sys.path.insert(0, '/content/demo2')

from model import GPT, GPTConfig
import tiktoken
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize


def format_story(paragraph, variant):
    """Format a plain-text story to match each model's training format."""
    if variant == 'forward':
        return paragraph
    # Split into 5 sentences for reverse/structured
    sents = sent_tokenize(paragraph.strip())
    sents = [s.strip() for s in sents if s.strip()]
    while len(sents) < 5:
        sents.append(sents[-1] if sents else '')
    sents = sents[:5]
    if variant == 'reverse':
        return ' '.join(reversed(sents))
    if variant == 'structured':
        return ' '.join('S' + str(i+1) + '> ' + s for i, s in enumerate(sents)).replace('S', '<S')
    return paragraph


def compute_ppl_on_eval_stories(out_dir, variant):
    """Load checkpoint and compute PPL on eval_stories.txt."""
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if not os.path.exists(ckpt_path):
        return None, None, 'checkpoint not found'

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    checkpoint = torch.load(ckpt_path, map_location=device)
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))

    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    prefix = '_orig_mod.'
    for k in list(state_dict.keys()):
        if k.startswith(prefix):
            state_dict[k[len(prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval().to(device)

    if device == 'cuda' and torch.cuda.is_bf16_supported():
        ctx = torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
    elif device == 'cuda':
        ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    else:
        ctx = nullcontext()

    enc = tiktoken.get_encoding('gpt2')
    encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})

    eval_path = 'data/rocstories/eval_stories.txt'
    if not os.path.exists(eval_path):
        del model
        return None, math.exp(best_val_loss), 'eval_stories.txt not found'

    text = open(eval_path, encoding='utf-8').read()
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    block_size = model.config.block_size

    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        with ctx:
            for para in paragraphs:
                formatted = format_story(para, variant)
                ids = encode(formatted)
                if len(ids) < 2:
                    continue
                for pos in range(0, len(ids) - 1, block_size):
                    inp = ids[pos: pos + block_size]
                    tgt = ids[pos + 1: pos + 1 + block_size]
                    if not tgt:
                        break
                    inp = inp[:len(tgt)]
                    x = torch.tensor(inp, dtype=torch.long, device=device)[None, :]
                    y = torch.tensor(tgt, dtype=torch.long, device=device)[None, :]
                    _, loss = model(x, y)
                    total_nll += loss.item() * len(tgt)
                    total_tokens += len(tgt)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if total_tokens == 0:
        return None, math.exp(best_val_loss), 'no tokens evaluated'
    return math.exp(total_nll / total_tokens), math.exp(best_val_loss), 'ok'


# ── Run evaluation ─────────────────────────────────────────────────────────
eval_models = [
    ('M1_BEST Forward',  'out-m1-best',       'forward',    'plain text'),
    ('M2 Reverse',       'out-m2-reverse',    'reverse',    'plain text reversed'),
    ('M3 Structured',    'out-m3-structured', 'structured', '<S1>...<S5> format'),
]

print(f'{"Model":<22} {"Eval format":<22} {"Val PPL":>9} {"Eval PPL":>9}')
print('-' * 68)

eval_results = {}
for name, out_dir, variant, fmt in eval_models:
    eval_ppl, val_ppl, status = compute_ppl_on_eval_stories(out_dir, variant)
    eval_results[name] = {'eval_ppl': eval_ppl, 'val_ppl': val_ppl}
    v_str = f'{val_ppl:.2f}'  if val_ppl  is not None else 'N/A'
    e_str = f'{eval_ppl:.2f}' if eval_ppl is not None else 'N/A'
    pass_flag = ''
    if eval_ppl is not None and variant in ('forward', 'reverse'):
        pass_flag = ' PASSES' if eval_ppl <= 25.0 else ' FAILS'
    print(f'{name:<22} {fmt:<22} {v_str:>9} {e_str:>9}{pass_flag}')

print('\nPPL ≤ 25.0 = passes tutor threshold (M1_BEST must pass)')

## Step 7: Generate Story Samples

In [ ]:
# ── Generate samples: M1_BEST Forward ────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')
os.makedirs('results/samples', exist_ok=True)

print('Generating 5 stories from M1_BEST (start=newline, plain text)...')
subprocess.run([
    sys.executable, 'sample.py',
    '--out_dir=out-m1-best',
    '--num_samples=5',
    '--max_new_tokens=150',
    '--temperature=0.8',
    '--top_k=100',
    '--save_to=results/samples/m1_best_samples.txt',
], capture_output=False)
print('Saved → results/samples/m1_best_samples.txt')

In [ ]:
# ── Generate samples: M2 Reverse ──────────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')

print('Generating 5 stories from M2 Reverse...')
print('NOTE: Stories read S5→S1 (resolution first) — this is expected.')
subprocess.run([
    sys.executable, 'sample.py',
    '--out_dir=out-m2-reverse',
    '--num_samples=5',
    '--max_new_tokens=150',
    '--temperature=0.8',
    '--top_k=100',
    '--save_to=results/samples/m2_reverse_samples.txt',
], capture_output=False)
print('Saved → results/samples/m2_reverse_samples.txt')

In [ ]:
# ── Generate samples: M3 Structured (free generation) ────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')

print('Generating 5 stories from M3 Structured (start=<S1>)...')
subprocess.run([
    sys.executable, 'sample.py',
    '--out_dir=out-m3-structured',
    '--start=<S1>',
    '--num_samples=5',
    '--max_new_tokens=150',
    '--temperature=0.8',
    '--top_k=100',
    '--save_to=results/samples/m3_structured_samples.txt',
], capture_output=False)
print('Saved → results/samples/m3_structured_samples.txt')

In [ ]:
# ── M3 Guided Story Completion (BONUS) ────────────────────────────────────
import os, subprocess, sys
os.chdir('/content/demo2')

print('M3 Guided Story Completion — give S1, model generates S2-S5')
print('Only possible with M3 due to <S1>...<S5> training format.')
print('=' * 60)

prompts = [
    '<S1> Mary wanted to do something special for her mom. <S2>',
    '<S1> John had always been afraid of dogs. <S2>',
    '<S1> The old car broke down on the highway. <S2>',
]

os.makedirs('results/samples', exist_ok=True)
with open('results/samples/m3_guided_completions.txt', 'w', encoding='utf-8') as out_f:
    for i, prompt in enumerate(prompts, 1):
        print(f'\n--- Prompt {i}: {prompt}')
        out_f.write(f'=== Prompt {i} ===\nGiven: {prompt}\n\n')
        result = subprocess.run([
            sys.executable, 'sample.py',
            '--out_dir=out-m3-structured',
            f'--start={prompt}',
            '--num_samples=2',
            '--max_new_tokens=120',
            '--temperature=0.8',
            '--top_k=100',
        ], capture_output=True, text=True)
        print(result.stdout)
        out_f.write(result.stdout + '\n')

print('\nSaved → results/samples/m3_guided_completions.txt')

## Step 8: Plot PPL Comparison

In [ ]:
import os, torch, math
import matplotlib.pyplot as plt

os.chdir('/content/demo2')

def get_best_val_ppl(out_dir):
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if not os.path.exists(ckpt_path):
        return None
    ckpt = torch.load(ckpt_path, map_location='cpu')
    return math.exp(ckpt.get('best_val_loss', float('inf')))

model_ppls = {
    'M1_BEST\nForward':    get_best_val_ppl('out-m1-best'),
    'M2\nReverse':         get_best_val_ppl('out-m2-reverse'),
    'M3\nStructured':      get_best_val_ppl('out-m3-structured'),
}

trained = {k: v for k, v in model_ppls.items() if v is not None}
print('Best validation PPL per model:')
for name, ppl in model_ppls.items():
    print(f'  {name.replace(chr(10), " "):<20} {f"{ppl:.2f}" if ppl else "not trained"}')

if trained:
    fig, ax = plt.subplots(figsize=(9, 5))
    names  = list(trained.keys())
    ppls   = list(trained.values())
    colors = ['#2196F3', '#F44336', '#4CAF50']

    bars = ax.bar(names, ppls, color=colors[:len(names)],
                  edgecolor='black', linewidth=0.7, width=0.5)

    # Tutor passing threshold
    ax.axhline(25.0, color='orange', linestyle='--', linewidth=2,
               label='PPL = 25.0 (tutor threshold)')

    for bar, ppl in zip(bars, ppls):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f'{ppl:.1f}', ha='center', va='bottom',
                fontweight='bold', fontsize=12)

    ax.set_ylabel('Best Validation Perplexity (lower = better)', fontsize=12)
    ax.set_title('M1_BEST vs M2 vs M3 — Perplexity Comparison', fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(ppls) * 1.3)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    os.makedirs('results', exist_ok=True)
    plt.tight_layout()
    plt.savefig('results/ppl_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/ppl_comparison.png')
else:
    print('No checkpoints found — run training first (Step 5).')

## Step 9: Results Summary Table

In [ ]:
import os, torch, math
os.chdir('/content/demo2')

model_info = [
    ('M1_BEST Forward (SUBMISSION)', 'out-m1-best',       'S1→S2→S3→S4→S5'),
    ('M2 Reverse',                   'out-m2-reverse',    'S5→S4→S3→S2→S1'),
    ('M3 Structured',                'out-m3-structured', '<Si> tags (NOT submitted)'),
]

print('=' * 80)
print('FINAL RESULTS TABLE')
print('=' * 80)
print(f'{"Model":<38} {"Format":<22} {"Best Val PPL":>13} {"Iter":>6}')
print('-' * 82)

rows = []
for name, out_dir, fmt in model_info:
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_ppl  = math.exp(ckpt.get('best_val_loss', float('inf')))
        iter_num = ckpt.get('iter_num', 0)
        print(f'{name:<38} {fmt:<22} {val_ppl:>13.2f} {iter_num:>6}')
        rows.append((name, fmt, val_ppl, iter_num))
    else:
        print(f'{name:<38} {fmt:<22} {"not trained":>13} {"":>6}')
        rows.append((name, fmt, None, None))

print('=' * 80)
print()
print('Hypotheses:')
print('  H1: M2 (Reverse) PPL > M1_BEST (Forward) PPL → causal asymmetry in narrative')
print('  H2: M3 (Structured, in-format) PPL ≤ M1_BEST PPL → explicit structure helps')
print()
ppls = [(n, p) for n, _, p, _ in rows if p is not None]
if len(ppls) >= 2:
    m1_ppl = next((p for n, p in ppls if 'BEST' in n or 'Forward' in n), None)
    m2_ppl = next((p for n, p in ppls if 'Reverse' in n), None)
    m3_ppl = next((p for n, p in ppls if 'Struct' in n), None)
    if m1_ppl and m2_ppl:
        h1 = 'CONFIRMED' if m2_ppl > m1_ppl else 'NOT confirmed'
        print(f'H1: {h1} — M2={m2_ppl:.2f} > M1={m1_ppl:.2f}')
    if m1_ppl and m3_ppl:
        h2 = 'CONFIRMED' if m3_ppl <= m1_ppl else 'NOT confirmed'
        print(f'H2: {h2} — M3={m3_ppl:.2f} vs M1={m1_ppl:.2f}')

# Save markdown
os.makedirs('results', exist_ok=True)
with open('results/perplexity_table.md', 'w', encoding='utf-8') as f:
    f.write('# Perplexity Results\n\n')
    f.write('| Model | Format | Best Val PPL | Final Iter |\n')
    f.write('|-------|--------|-------------|-----------|\n')
    for name, fmt, val_ppl, iter_num in rows:
        p_str = f'{val_ppl:.2f}' if val_ppl else 'N/A'
        i_str = str(iter_num) if iter_num else 'N/A'
        f.write(f'| {name} | {fmt} | {p_str} | {i_str} |\n')
print('\nSaved: results/perplexity_table.md')

## Step 10: Upload M1_BEST to HuggingFace (Task 3 Submission)

**Submit M1_BEST (plain-text model) — NOT M3.**  
M3 fails tutor plain-text evaluation with PPL ~94. M1_BEST is consistent with tutor evaluation.

`sampling_params.json` uses `start='\n'` (plain text prompt, not `<S1>`).

In [ ]:
# ── Create sampling_params.json for M1_BEST ───────────────────────────────
import json, os, shutil
os.chdir('/content/demo2')

# IMPORTANT: start='\n' for M1_BEST (plain text).
# Do NOT use '<S1>' here — that is M3's format and would confuse M1_BEST.
sampling_params = {
    'temperature': 0.8,
    'top_k': 100,
    'max_new_tokens': 200,
    'start': '\n'
}

os.makedirs('out-m1-best', exist_ok=True)
with open('out-m1-best/sampling_params.json', 'w') as f:
    json.dump(sampling_params, f, indent=2)

print('Created: out-m1-best/sampling_params.json')
print(json.dumps(sampling_params, indent=2))

# Copy model.py to checkpoint dir (required for submission)
shutil.copy('model.py', 'out-m1-best/model.py')
print('\nCopied: model.py → out-m1-best/model.py')

print('\nSubmission directory contents:')
for fname in os.listdir('out-m1-best'):
    path = os.path.join('out-m1-best', fname)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f'  {fname:<35} ({size/1e6:.2f} MB)')

In [ ]:
# ── Upload to HuggingFace ─────────────────────────────────────────────────
# Replace with your HuggingFace username and token
HF_USERNAME = 'YOUR_HF_USERNAME'
HF_TOKEN    = 'hf_YOUR_TOKEN_HERE'
REPO_NAME   = 'nanoGPT_hw'

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

from huggingface_hub import HfApi
import os
os.chdir('/content/demo2')

api = HfApi()
repo_id = f'{HF_USERNAME}/{REPO_NAME}'

# Create public repo
try:
    api.create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True, private=False)
    print(f'Repo ready: https://huggingface.co/{repo_id}')
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload required files — M1_BEST (plain text submission)
upload_files = [
    ('out-m1-best/ckpt.pt',              'ckpt.pt'),
    ('out-m1-best/model.py',             'model.py'),
    ('out-m1-best/sampling_params.json', 'sampling_params.json'),
]

for local_path, repo_path in upload_files:
    if os.path.exists(local_path):
        print(f'Uploading {local_path} → {repo_path} ...')
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=repo_id,
            token=HF_TOKEN,
        )
        print(f'  Uploaded: {repo_path}')
    else:
        print(f'  NOT FOUND: {local_path}')

print(f'\nSubmission URL: https://huggingface.co/{repo_id}')
print('Submit this URL + your HF token to Canvas.')

---

## Quick Reference: All Commands

```bash
# ─── Data preparation ───────────────────────────────────────────────────────
python data/rocstories/prepare.py --variant=forward      # M1_BEST
python data/rocstories/prepare.py --variant=reverse      # M2
python data/rocstories/prepare.py --variant=structured   # M3
python data/rocstories/prepare.py --variant=all          # All 3

# ─── Smoke tests (50 iterations) ────────────────────────────────────────────
python train.py config/test_m1_best.py
python train.py config/test_m2_reverse.py
python train.py config/test_m3_structured.py

# ─── Full training (10 000 iterations) ──────────────────────────────────────
python train.py config/train_m1_best.py       # SUBMISSION MODEL
python train.py config/train_m2_reverse.py
python train.py config/train_m3_structured.py

# ─── Generate samples ───────────────────────────────────────────────────────
python sample.py --out_dir=out-m1-best --num_samples=5 --max_new_tokens=150
python sample.py --out_dir=out-m2-reverse --num_samples=5 --max_new_tokens=150
python sample.py --out_dir=out-m3-structured --start='<S1>' --num_samples=5

# M3 guided completion:
python sample.py --out_dir=out-m3-structured \
    --start='<S1> Mary went to school. <S2>' --num_samples=3
```

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size=32` in config, restart runtime |
| `No module named tiktoken` | Run `pip install tiktoken` |
| `CSV not found` | Run Step 3 again |
| `ckpt.pt not found` | Training didn't save — check `always_save_checkpoint` or val loss never improved |
| Compilation error (`torch.compile`) | Add `compile=False` to config |
| Session crashed, no checkpoint | Checkpoints are on Drive — re-run Step 2 symlinks, resume |
| M1 PPL > 25.0 | Run `train_m1_best.py` again (10 000 iters should fix it) |
| M3 PPL high on plain text | Expected — M3 is NOT the submission model |